This notebook is an initial notebook aimed at understanding the data, properly cleaning it and reducing it's size before applying large NLP manipulations on it.

The output dataset here is used as the full dataset on which stance detection model will be applied.

In [ ]:
import pandas as pd
import re

In [12]:
# Load the dataset
df = pd.read_csv("Data and models/reddit_opinion_PSE_ISR.csv", encoding='utf-8')

# Display basic info
print("Shape:", df.shape)
print("\nColumn Names:", df.columns.tolist())

# Show a small sample
display(df.sample(5))

# Check for nulls
print("\nMissing values per column:\n", df.isnull().sum())

# Check unique values for short categorical columns
for col in df.columns:
    if df[col].nunique() < 20:
        print(f"\nColumn: {col} | Unique values:", df[col].unique())

Shape: (2767778, 24)

Column Names: ['comment_id', 'score', 'self_text', 'subreddit', 'created_time', 'post_id', 'author_name', 'controversiality', 'ups', 'downs', 'user_is_verified', 'user_account_created_time', 'user_awardee_karma', 'user_awarder_karma', 'user_link_karma', 'user_comment_karma', 'user_total_karma', 'post_score', 'post_self_text', 'post_title', 'post_upvote_ratio', 'post_thumbs_ups', 'post_total_awards_received', 'post_created_time']


,comment_id,score,self_text,subreddit,created_time,post_id,author_name,controversiality,ups,downs,...,user_link_karma,user_comment_karma,user_total_karma,post_score,post_self_text,post_title,post_upvote_ratio,post_thumbs_ups,post_total_awards_received,post_created_time
817958,lpnrebc,1,Sweet dreams Nassrallah.,CombatFootage,2024-09-30 15:08:43,1fss4uf,Fr33speechisdeAd,0,1,0,...,14.0,14176.0,14190.0,1902,NaN,"Another angle of IDF airstrike in beirut, 28 S...",0.95,1902,0,2024-09-30 10:17:33
1727036,kvvmcnj,16,same way that jews who got kicked out of arab ...,IsraelPalestine,2024-03-21 12:19:58,1bk068w,saargrin,0,16,0,...,38083.0,126210.0,165114.0,68,UNRWA's [defines](https://www.unrwa.org/sites/...,Can Israeli Jews register as Palestinian refug...,0.82,68,0,2024-03-21 06:38:13
843315,loxlnyd,41,They still have nuclear and hydro powerplants..,worldnews,2024-09-25 22:49:55,1fpeo05,DownvoteEvangelist,0,41,0,...,2149.0,259715.0,261864.0,26203,NaN,Russia has destroyed all thermal power plants ...,0.95,26203,0,2024-09-25 20:37:53
2734661,k7k14em,-5,Lmao if you think i am mad you are wrong bro j...,PublicFreakout,2023-11-02 20:23:08,17m5og1,TheAngryXennial,0,-5,0,...,452.0,11604.0,12056.0,4872,NaN,Let's talk about Israel,0.93,4872,0,2023-11-02 15:11:14
133761,mr8ejrg,8,Your source is an allegation just like these a...,IsraelPalestine,2025-05-08 12:22:33,1khmt30,InevitableHome343,0,8,0,...,724.0,64538.0,65262.0,22,"""...In April 2024, Samantha Power, director of...",The Gaza famine myth,0.57,22,0,2025-05-08 10:33:05



Missing values per column:
 comment_id                          0
score                               0
self_text                          14
subreddit                           0
created_time                        0
post_id                             0
author_name                         0
controversiality                    0
ups                                 0
downs                               0
user_is_verified                    0
user_account_created_time      121538
user_awardee_karma                405
user_awarder_karma                405
user_link_karma                   405
user_comment_karma                405
user_total_karma                  405
post_score                          0
post_self_text                1514453
post_title                          0
post_upvote_ratio                   0
post_thumbs_ups                     0
post_total_awards_received          0
post_created_time                   0
dtype: int64

Column: subreddit | Unique values: ['worldnew

In [ ]:
# Select relevant columns
columns_to_keep = [
    'comment_id', 'score', 'self_text', 'subreddit', 'created_time',
    'post_id', 'author_name', 'ups', 'downs', 'post_score',
    'post_self_text', 'post_title', 'post_created_time'
]
df = df[columns_to_keep].copy()

# Patterns
USER_PATTERN = re.compile(r"(u\/[a-zA-Z0-9_-]+)")
MEDIA_PATTERN = re.compile(r"(https?:\/\/\S+\.(jpg|jpeg|png|gif|webm|mp4|mov))", re.IGNORECASE)
URL_PATTERN = re.compile(r"(https?:\/\/\S+)")
HTML_ARTIFACTS = {"&gt;": ">", "&lt;": "<", "&amp;": "&", "&#x200B;": " "}
UNUSABLE_STRINGS = ["[deleted]", "[removed]", ""]

# General text cleaning (no mention or media extraction)
def basic_clean(text):
    if not isinstance(text, str):
        return ""
    text = text.strip()
    if text.lower() in UNUSABLE_STRINGS or len(text) < 3:
        return ""
    for bad, good in HTML_ARTIFACTS.items():
        text = text.replace(bad, good)
    text = re.sub(r"^>.*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"[^\w\s.,!?'\"]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# Full clean with mention/link/media processing (for self_text only)
def clean_self_text(text):
    if not isinstance(text, str):
        return "", [], []
    text = text.strip()
    if text.lower() in UNUSABLE_STRINGS or len(text) < 3:
        return "", [], []
    for bad, good in HTML_ARTIFACTS.items():
        text = text.replace(bad, good)
    mentions = USER_PATTERN.findall(text)
    media_links = MEDIA_PATTERN.findall(text)
    text = MEDIA_PATTERN.sub("[MEDIA]", text)
    text = URL_PATTERN.sub("[LINK]", text)
    text = re.sub(r"^>.*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"[^\w\s.,!?'\"]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip(), mentions, media_links

# Apply post_self_text basic cleaning
print("Cleaning post_self_text...")
df["post_self_text"] = df["post_self_text"].apply(basic_clean)

# Apply self_text full cleaning
print("Cleaning self_text...")
df[["self_text", "user_mentions", "media_links"]] = df["self_text"].apply(
    lambda t: pd.Series(clean_self_text(t))
)

# Drop rows with unusable comment body
df = df[
    df["self_text"].str.len() >= 10
].copy()

# Drop rows with nulls in all columns *except* post_self_text
df["created_time"] = pd.to_datetime(df["created_time"], errors='coerce')
columns_required = [col for col in df.columns if col != "post_self_text"]
clean_df = df.dropna(subset=columns_required).copy()

# Format dates consistently (e.g., ISO format)
clean_df["created_time"] = clean_df["created_time"].dt.strftime("%Y-%m-%d %H:%M:%S")

Cleaning post_self_text...
Cleaning self_text...


In [16]:
# Build set of known authors (lowercased for matching)
known_users = set(clean_df["author_name"].str.lower().unique())

# Normalize mentions by stripping 'u/' and lowercasing
def normalize_mentions(mention_list):
    if not isinstance(mention_list, list):
        return []
    return [m[2:].lower() for m in mention_list if isinstance(m, str) and m.startswith("u/")]

# Filter mentions
clean_df["user_mentions"] = clean_df["user_mentions"].apply(
    lambda lst: [m for m in normalize_mentions(lst) if m in known_users]
)

In [18]:
# Final save
output_path = "Data and models/reddit_opinion_PSE_ISR_cleaned.csv"
clean_df.to_csv(output_path, index=False)

In [19]:
# Stats
num_comments = len(clean_df)
num_unique_users = clean_df["author_name"].nunique()
num_unique_posts = clean_df["post_id"].nunique()
num_mentions = clean_df["user_mentions"].explode().dropna().nunique()

# Print statistics
print("Cleaned comments:", num_comments)
print("Unique users:", num_unique_users)
print("Unique posts:", num_unique_posts)
print("Unique user mentions:", num_mentions)
print("Saved to:", output_path)

Cleaned comments: 2697930
Unique users: 377272
Unique posts: 52378
Unique user mentions: 5363
Saved to: Data and models/reddit_opinion_PSE_ISR_cleaned.csv


In [20]:
clean_df.head()

,comment_id,score,self_text,subreddit,created_time,post_id,author_name,ups,downs,post_score,post_self_text,post_title,post_created_time,user_mentions,media_links
0,mwgtiz7,1,"Again though, his actions are forced by member...",worldnews,2025-06-07 10:09:24,1l5d15g,nhytgbvfeco,1,0,863,,"Israel: Bennett's New Party Leads in Polls, Ne...",2025-06-07 05:12:01,[],[]
1,mwgtip7,1,The German police arresting a Jewish man for a...,Palestine,2025-06-07 10:09:19,1l5g9n3,DmeshOnPs5,1,0,232,,During a protest organized by the Left Party i...,2025-06-07 08:49:57,[],[]
2,mwgthp6,1,!gifgiphy3o7aTuFSbugZXppHb2,PublicFreakout,2025-06-07 10:09:03,1l5gwwp,WorldRecordOnline,1,0,28,Germany hasn't abandoned nazism,During a protest organized by the Left Party i...,2025-06-07 09:34:46,[],[]
3,mwgten5,1,We're already there. The killing just isn't en...,PublicFreakout,2025-06-07 10:08:13,1l5cxvw,HailLugalKiEn,1,0,1251,,All Eyes on are on the feds: Feds Terrorize L....,2025-06-07 05:06:26,[],[]
4,mwgte3r,1,Thats not the definition of tokenism. That is ...,IsraelPalestine,2025-06-07 10:08:03,1l53z09,Agreeable_Recipe3075,1,0,0,Ive been thinking a lot about what it means to...,I don’t care about a Jewish State. I want Isra...,2025-06-06 21:26:56,[],[]


In [22]:
clean_df_c = clean_df[["comment_id", "self_text"]].copy()
clean_df_c["final_label"] = 0
clean_df_c["subset"] = "TEST"

print("Saving data...")
output_path = "Data and models/reddit_opinion_PSE_ISR_cleaned_for_embedding.csv"
clean_df_c.to_csv(output_path, index=False)

Saving data...
